# Generate Graph-Spectral Diagnostics for MultiUN Translations

This Generate notebook analyzes cached [MultiUN](https://huggingface.co/datasets/Helsinki-NLP/multiun) samples shown in `Demo-multiun-translation-samples.ipynb` and computes Llama-3.2-3B diagnostics for **Preliminary Graph-Spectral Comparison of Numerical, Natural-Language, and Code Inputs** in the appendix. It provides the results for the six-language United Nations translation comparison.

The notebook analyzes 50 samples per language at context lengths from 100 to 2,000 tokens. It measures attention-graph connectivity and the distribution of hidden-state energy across graph Fourier modes. Metrics are averaged over all 28 layers, then summarized across samples.


Set model, graph, spectral, and checkpoint options, relative paths, and random seeds.

Run this notebook with `MultiUN-translations` as the working directory. Cached inputs are read from `multiun_language_samples`. Checkpoints stay in the local `cache/`; final diagnostic results are saved directly to the shared appendix `../../cache/` used by the visualization notebook.


In [ ]:
import os
from pathlib import Path

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
import gc
import random
from pathlib import Path

import numpy as np
import torch
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

# Model identity and reproducibility
SEED = 42
MODEL_NAME = "meta-llama/Llama-3.2-3B"
EXPECTED_NUM_LAYERS = 28


# Prepend one BOS token for inference (Llama-3.2-3B tokenizer default).
# Exclude its position from spectral diagnostics.
DROP_BOS = True
PREPEND_BOS = True
RENORMALIZE_AFTER_DROP = True
HEAD_WEIGHTING = "mass"
TOKENIZATION_ADD_SPECIAL_TOKENS = False
K_CUTOFF = None

# Sweep size and checkpoint controls
RESUME_FROM_CHECKPOINT = True
CHECKPOINT_EVERY_N_TASKS = 200
EXPECTED_N_SAMPLES_PER_FAMILY = 50
RUN_SUBSAMPLE_SIZE = None  # Use all saved runs.
RUN_SUBSAMPLE_SEED = SEED
# Keep checkpoints local; save final results in the shared appendix cache.
PROJECT_DIR = Path(".")
DATA_DIR = PROJECT_DIR / "multiun_language_samples"
APPENDIX_DIR = PROJECT_DIR / ".." / ".."
LOCAL_CACHE_DIR = PROJECT_DIR / "cache"
SHARED_CACHE_DIR = APPENDIX_DIR / "cache"
LOCAL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
SHARED_CACHE_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_NPZ_PATH = DATA_DIR / "multiun_language_metric_inputs_2000_tokens.npz"

OUTPUT_PATH = SHARED_CACHE_DIR / "multiun_language_all_metrics_averaged_over_layers_vs_context_length.npz"
CHECKPOINT_PATH = LOCAL_CACHE_DIR / "multiun_language_all_metrics_averaged_over_layers_vs_context_length.50runs.5pct.checkpoint.npz"

# Metric names shared with downstream visualization

layer_averaged_metric_names = np.asarray([
    "layer_averaged_effective_active_modes",
    "layer_averaged_normalized_fiedler_value_lambda_2",
    "layer_averaged_hfer",
], dtype=str)
metric_names = layer_averaged_metric_names.copy()

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

Load and validate saved token samples, then select the language, sample, and context axes.


In [ ]:
with np.load(SAMPLE_NPZ_PATH, allow_pickle=False) as data:
    token_ids_full = data["token_ids"].astype(np.int64)
    input_data_order_full = data["input_names"].astype(str)
    run_ids_full = data["run_ids"].astype(np.int64)
    context_token_counts_full = data["context_token_counts"].astype(np.int64)
    context_fracs_full = data["context_fracs"].astype(np.float64)
    k_cutoff_by_context_full = data["k_cutoff_by_context"].astype(np.int64)

    sample_model_name = str(np.asarray(data["model_name"]).item())
    tokenizer_name = str(np.asarray(data["tokenizer_name"]).item())
    max_context_tokens = int(np.asarray(data["max_context_tokens"]).item())
    source_n_samples_per_family = int(np.asarray(data["n_samples_per_family"]).item())
    sample_schema_version = str(np.asarray(data["schema_version"]).item())
    sample_construction_mode = str(np.asarray(data["sample_construction_mode"]).item())

    sample_bos_token_id = (
        int(np.asarray(data["bos_token_id"]).item())
        if "bos_token_id" in data.files
        else None
    )
    sample_eos_token_id = (
        int(np.asarray(data["eos_token_id"]).item())
        if "eos_token_id" in data.files
        else None
    )
    family_condition_types_full = (
        data["family_condition_types"].astype(str)
        if "family_condition_types" in data.files
        else input_data_order_full.copy()
    )
    language_codes_full = (
        data["language_codes"].astype(str)
        if "language_codes" in data.files
        else input_data_order_full.copy()
    )
    language_names_full = (
        data["language_names"].astype(str)
        if "language_names" in data.files
        else input_data_order_full.copy()
    )

# Check the input construction, model identity, and array dimensions.
if "shuffled" in sample_schema_version.lower() or "shuffled" in sample_construction_mode.lower():
    raise ValueError("This notebook must analyze only original-order MultiUN language samples, not shuffled controls.")

if sample_model_name != MODEL_NAME:
    raise ValueError(f"Sample model_name={sample_model_name!r} does not match MODEL_NAME={MODEL_NAME!r}.")

if tokenizer_name != MODEL_NAME:
    raise ValueError(f"Sample tokenizer_name={tokenizer_name!r} does not match MODEL_NAME={MODEL_NAME!r}.")

if token_ids_full.ndim != 3:
    raise ValueError(f"Expected token_ids to be 3D, got shape {token_ids_full.shape}.")

if token_ids_full.shape[0] != len(input_data_order_full):
    raise ValueError("token_ids first axis must match input_names.")

if token_ids_full.shape[1] != len(run_ids_full):
    raise ValueError("token_ids second axis must match run_ids.")

if token_ids_full.shape[1] != source_n_samples_per_family:
    raise ValueError("token_ids second axis must match n_samples_per_family.")
if source_n_samples_per_family != EXPECTED_N_SAMPLES_PER_FAMILY:
    raise ValueError(
        f"Expected {EXPECTED_N_SAMPLES_PER_FAMILY} saved runs per language, "
        f"got {source_n_samples_per_family}."
    )

if token_ids_full.shape[2] != max_context_tokens:
    raise ValueError("token_ids third axis must match max_context_tokens.")

if family_condition_types_full.shape[0] != input_data_order_full.shape[0]:
    raise ValueError("family_condition_types must align with input_names.")
if language_codes_full.shape[0] != input_data_order_full.shape[0]:
    raise ValueError("language_codes must align with input_names.")
if language_names_full.shape[0] != input_data_order_full.shape[0]:
    raise ValueError("language_names must align with input_names.")

if int(context_token_counts_full[-1]) != int(max_context_tokens):
    raise ValueError("Final context length must equal max_context_tokens.")

# Require the 100-token grid and its default halfway spectral cutoff.
expected_context_token_counts = np.arange(100, max_context_tokens + 1, 100, dtype=np.int64)
if not np.array_equal(context_token_counts_full, expected_context_token_counts):
    raise ValueError(
        "context_token_counts must be the 5% 100-token grid ending at max_context_tokens."
    )

expected_context_fracs = context_token_counts_full.astype(np.float64) / float(max_context_tokens)
if not np.allclose(context_fracs_full, expected_context_fracs):
    raise ValueError("context_fracs must equal context_token_counts / max_context_tokens.")

if not np.array_equal(k_cutoff_by_context_full, context_token_counts_full // 2):
    raise ValueError("k_cutoff_by_context must equal context_token_counts // 2.")

if not np.all(np.diff(context_token_counts_full) > 0):
    raise ValueError("context_token_counts must be strictly increasing.")

if np.any(token_ids_full < 0):
    raise ValueError("Saved token IDs must be nonnegative.")

# Select all saved inputs and contexts
run_subsample_seed = -1

active_input_indices = np.arange(input_data_order_full.size, dtype=np.int64)
if RUN_SUBSAMPLE_SIZE is None:
    active_run_indices = np.arange(run_ids_full.size, dtype=np.int64)
else:
    requested_run_subsample_size = int(RUN_SUBSAMPLE_SIZE)
    if requested_run_subsample_size <= 0 or requested_run_subsample_size > run_ids_full.size:
        raise ValueError(
            f"RUN_SUBSAMPLE_SIZE must be in [1, {run_ids_full.size}], "
            f"got {requested_run_subsample_size}."
        )
    run_subsample_seed = int(RUN_SUBSAMPLE_SEED)
    run_rng = np.random.default_rng(run_subsample_seed)
    active_run_indices = np.sort(
        run_rng.choice(run_ids_full.size, size=requested_run_subsample_size, replace=False).astype(np.int64)
    )
active_context_indices = np.arange(context_token_counts_full.size, dtype=np.int64)

input_names = input_data_order_full[active_input_indices]
family_condition_types = family_condition_types_full[active_input_indices]
language_codes = language_codes_full[active_input_indices]
language_names = language_names_full[active_input_indices]
run_ids = run_ids_full[active_run_indices]
n_samples_per_family = int(len(run_ids))
run_subsample_size = int(len(run_ids))
context_token_counts = context_token_counts_full[active_context_indices]
context_fracs = context_fracs_full[active_context_indices]
k_cutoff_by_context = k_cutoff_by_context_full[active_context_indices]

token_ids = token_ids_full[
    np.ix_(active_input_indices, active_run_indices, np.arange(token_ids_full.shape[2]))
]

if token_ids.shape[:2] != (len(input_names), len(run_ids)):
    raise ValueError("Selected token_ids must align with active input_names and run_ids.")


print(
    f"Loaded MultiUN language samples: {len(input_names)} families x {len(run_ids)} samples x "
    f"{len(context_token_counts)} contexts; token_ids shape={token_ids.shape}."
)


Validate the tokenizer and build model inputs directly from saved token IDs with one prepended BOS token.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

# Confirm that the tokenizer special-token IDs agree with the sample archive.
if tokenizer.bos_token_id is None:
    raise ValueError("Tokenizer must define a BOS token ID.")
if sample_bos_token_id is not None and int(tokenizer.bos_token_id) != int(sample_bos_token_id):
    raise ValueError(
        f"Tokenizer BOS token ID {tokenizer.bos_token_id} does not match sample BOS token ID {sample_bos_token_id}."
    )
if sample_eos_token_id is not None and tokenizer.eos_token_id is not None:
    if int(tokenizer.eos_token_id) != int(sample_eos_token_id):
        raise ValueError(
            f"Tokenizer EOS token ID {tokenizer.eos_token_id} does not match sample EOS token ID {sample_eos_token_id}."
        )


def _first_token_value(x):
    if x is None:
        return None
    if isinstance(x, torch.Tensor):
        if x.numel() == 0:
            return None
        return int(x.flatten()[0].item())
    arr = np.asarray(x).reshape(-1)
    if arr.size == 0:
        return None
    return int(arr[0])


def get_token_start_index(
    input_ids=None,
    bos_token_id=None,
    special_tokens_mask=None,
    drop_bos=True,
):
    if not drop_bos:
        return 0

    first_id = _first_token_value(input_ids)
    if first_id is not None and bos_token_id is not None and first_id == int(bos_token_id):
        return 1

    first_mask = _first_token_value(special_tokens_mask)
    if first_mask == 1:
        return 1

    return 0


def build_model_inputs_from_saved_token_ids(
    sample_token_ids,
    context_token_count,
    tokenizer,
    device,
    prepend_bos=True,
):
    sample_token_ids = np.asarray(sample_token_ids, dtype=np.int64)
    context_token_count = int(context_token_count)

    if sample_token_ids.ndim != 1:
        raise ValueError(f"Expected one-dimensional sample_token_ids, got {sample_token_ids.shape}.")

    if sample_token_ids.size < context_token_count:
        raise ValueError(
            f"Need at least {context_token_count} saved token IDs, got {sample_token_ids.size}."
        )

    # Use a prefix of the saved IDs directly; do not decode and retokenize it.
    content_ids = sample_token_ids[:context_token_count]
    if np.any(content_ids < 0):
        raise ValueError("Content token IDs contain negative values.")

    if prepend_bos and content_ids.size > 0 and int(content_ids[0]) == int(tokenizer.bos_token_id):
        raise ValueError("Saved content token IDs already begin with BOS; refusing to prepend a second BOS.")

    # Prepend exactly one BOS token and mark it for removal before metric computation.
    if prepend_bos:
        if tokenizer.bos_token_id is None:
            raise ValueError("PREPEND_BOS=True but tokenizer.bos_token_id is None.")
        full_ids = np.concatenate(
            [np.asarray([int(tokenizer.bos_token_id)], dtype=np.int64), content_ids],
            axis=0,
        )
        special_tokens_mask = np.concatenate(
            [np.asarray([1], dtype=np.int64), np.zeros(context_token_count, dtype=np.int64)],
            axis=0,
        )
    else:
        full_ids = content_ids
        special_tokens_mask = np.zeros(context_token_count, dtype=np.int64)

    input_ids = torch.tensor(full_ids[None, :], dtype=torch.long, device=device)
    attention_mask = torch.ones_like(input_ids, dtype=torch.long, device=device)
    special_tokens_mask_tensor = torch.tensor(
        special_tokens_mask,
        dtype=torch.long,
        device=device,
    )

    token_start = get_token_start_index(
        input_ids=input_ids[0],
        bos_token_id=tokenizer.bos_token_id,
        special_tokens_mask=special_tokens_mask_tensor,
        drop_bos=DROP_BOS,
    )
    # Context length counts content tokens, excluding the prepended BOS token.
    post_bos_len = int(input_ids.shape[1]) - int(token_start)

    if post_bos_len != context_token_count:
        raise ValueError(
            f"Expected {context_token_count} post-BOS content tokens, got {post_bos_len}."
        )

    return {
        "model_inputs": {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
        },
        "input_ids_for_metrics": input_ids[0],
        "special_tokens_mask": special_tokens_mask_tensor,
        "token_start": int(token_start),
        "seq_len": int(post_bos_len),
    }


for context_idx, context_token_count in enumerate(context_token_counts):
    if int(k_cutoff_by_context[context_idx]) != int(context_token_count) // 2:
        raise ValueError("k_cutoff_by_context must equal context_token_counts // 2.")


Construct attention graphs and compute hidden-state energy diagnostics and the normalized Fiedler value.


In [ ]:
def compute_all_layer_metrics_from_attention_and_hidden(
    attention_layer,
    hidden_state_layer,
    input_ids,
    layer_idx,
    expected_seq_len,
    k_cutoff=None,
    bos_token_id=None,
    special_tokens_mask=None,
    drop_bos=True,
    renormalize_after_drop=True,
    head_weighting="mass",
):
    token_start = get_token_start_index(
        input_ids=input_ids,
        bos_token_id=bos_token_id,
        special_tokens_mask=special_tokens_mask,
        drop_bos=drop_bos,
    )

    # Remove BOS from both attention axes to keep only content-token graph nodes.
    A = attention_layer[0].float()
    A = A[:, token_start:, token_start:]
    seq_len = int(A.shape[-1])
    if seq_len != int(expected_seq_len):
        raise ValueError(f"Layer {layer_idx}: expected seq_len={expected_seq_len}, got {seq_len}.")
    if seq_len < 2:
        raise ValueError(f"Layer {layer_idx}: lambda_2 requires at least two graph nodes, got {seq_len}.")

    # Restore row sums after removing attention to the BOS token.
    if renormalize_after_drop and token_start > 0:
        row_sums = A.sum(dim=-1, keepdim=True)
        row_sums = torch.where(row_sums == 0, torch.ones_like(row_sums), row_sums)
        A = A / row_sums

    if k_cutoff is None:
        k_cutoff = seq_len // 2
    k_cutoff = int(k_cutoff)
    if k_cutoff < 0 or k_cutoff > seq_len:
        raise ValueError(f"Layer {layer_idx}: invalid k_cutoff={k_cutoff} for seq_len={seq_len}.")

    # Symmetrize each head, then combine its undirected weights with the selected head weights.
    W_heads = 0.5 * (A + A.transpose(-1, -2))
    if head_weighting == "uniform":
        alpha = torch.full((A.shape[0],), 1.0 / A.shape[0], device=A.device, dtype=A.dtype)
    elif head_weighting == "mass":
        # A row-normalized head has total mass equal to the node count, giving uniform weights.
        s = A.sum(dim=(1, 2))
        s_sum = s.sum()
        if s_sum <= 0:
            alpha = torch.full((A.shape[0],), 1.0 / A.shape[0], device=A.device, dtype=A.dtype)
        else:
            alpha = s / s_sum
    else:
        raise ValueError("head_weighting must be 'mass' or 'uniform'.")

    Wbar = (alpha[:, None, None] * W_heads).sum(dim=0)
    deg = Wbar.sum(dim=1)

    # Use the combinatorial Laplacian eigenvectors as the graph Fourier basis.
    L = torch.diag(deg) - Wbar
    _, U = torch.linalg.eigh(L)

    # Align the uncentered hidden-state signal with the same content-token nodes.
    X = hidden_state_layer[0].float()
    X = X[token_start:, :].to(U.device)
    if X.shape[0] != seq_len:
        raise ValueError(
            f"Layer {layer_idx}: hidden-state length mismatch after BOS drop; X={X.shape[0]}, graph={seq_len}."
        )

    # Sum squared Fourier coefficients over hidden features to obtain energy per mode.
    X_hat = U.T @ X
    energy = (X_hat ** 2).sum(dim=1)
    total_energy = energy.sum()
    if total_energy <= 0 or not torch.isfinite(total_energy):
        raise ValueError(f"Layer {layer_idx}: nonpositive or nonfinite total signal energy: {total_energy}.")
    p = energy / total_energy
    positive_p = p[p > 0]
    # Exponentiated entropy measures how many modes effectively carry the signal energy.
    spectral_entropy = -torch.sum(positive_p * torch.log(positive_p))
    effective_active_modes = torch.exp(spectral_entropy)

    # HFER sums energy from k_cutoff onward; saved contexts use the halfway cutoff.
    hfer = float(p[k_cutoff:].sum().detach().cpu().item())


    # Use the symmetric normalized Laplacian for the Fiedler value, lambda_2.
    inv_sqrt_deg = torch.where(deg > 0, deg.pow(-0.5), torch.zeros_like(deg))
    Wbar_norm = inv_sqrt_deg[:, None] * Wbar * inv_sqrt_deg[None, :]
    L_sym = torch.eye(seq_len, device=Wbar.device, dtype=Wbar.dtype) - Wbar_norm
    eigvals_sym = torch.linalg.eigvalsh(L_sym)

    result = {
        "effective_active_modes": float(effective_active_modes.detach().cpu().item()),
        "normalized_fiedler_value": float(eigvals_sym[1].detach().cpu().item()),
        "hfer": hfer,
        "seq_len": int(seq_len),
        "k_cutoff": int(k_cutoff),
        "token_start": int(token_start),
        "attention_layer_index": int(layer_idx),
        "signal_layer_index": int(layer_idx),
    }

    del A, W_heads, Wbar, deg, L, U, X, X_hat, energy, p, inv_sqrt_deg, Wbar_norm, L_sym, eigvals_sym
    return result


Define result arrays and save/load helpers for resuming a compatible checkpoint.


In [ ]:
def _expected_result_shape():
    # Metrics use [language, sample, context].
    base_shape = (len(input_names), len(run_ids), len(context_token_counts))
    return base_shape


def _initialize_result_arrays():
    base_shape = _expected_result_shape()
    return {
        "active_modes_layeravg_per_run": np.full(base_shape, np.nan, dtype=np.float64),
        "fiedler_layeravg_per_run": np.full(base_shape, np.nan, dtype=np.float64),
        "hfer_layeravg_per_run": np.full(base_shape, np.nan, dtype=np.float64),
        "completed_mask": np.zeros(base_shape, dtype=bool),
    }


def _npz_scalar_to_str(data, key):
    return str(np.asarray(data[key]).item())


# Check required fields, source, model, selected axes, and shapes before resuming.
def _validate_checkpoint_compatibility(data, num_layers):
    required_fields = {
        "active_modes_layeravg_per_run",
        "completed_mask",
        "context_fracs",
        "context_token_counts",
        "data_ordering",
        "family_condition_types",
        "fiedler_layeravg_per_run",
        "hfer_layeravg_per_run",
        "input_names",
        "k_cutoff_by_context",
        "max_context_tokens",
        "model_name",
        "n_samples_per_family",
        "num_layers",
        "run_ids",
        "run_subsample_seed",
        "run_subsample_size",
        "sample_construction_mode",
        "sample_schema_version",
        "source_n_samples_per_family",
        "source_sample_npz_path",
        "tokenizer_name",
    }
    missing_fields = required_fields - set(data.files)
    if missing_fields:
        raise ValueError(f"Checkpoint is missing required fields: {sorted(missing_fields)}.")

    if _npz_scalar_to_str(data, "model_name") != MODEL_NAME:
        raise ValueError("Checkpoint model_name mismatch.")
    if os.path.relpath(_npz_scalar_to_str(data, "source_sample_npz_path")) != str(SAMPLE_NPZ_PATH):
        raise ValueError("Checkpoint source_sample_npz_path mismatch.")
    if _npz_scalar_to_str(data, "sample_schema_version") != sample_schema_version:
        raise ValueError("Checkpoint sample_schema_version mismatch.")
    if _npz_scalar_to_str(data, "sample_construction_mode") != sample_construction_mode:
        raise ValueError("Checkpoint sample_construction_mode mismatch.")
    if _npz_scalar_to_str(data, "tokenizer_name") != tokenizer_name:
        raise ValueError("Checkpoint tokenizer_name mismatch.")
    if _npz_scalar_to_str(data, "data_ordering") != "original":
        raise ValueError("Checkpoint data_ordering mismatch.")
    if int(np.asarray(data["num_layers"]).item()) != int(num_layers):
        raise ValueError("Checkpoint num_layers mismatch.")
    if int(np.asarray(data["max_context_tokens"]).item()) != int(max_context_tokens):
        raise ValueError("Checkpoint max_context_tokens mismatch.")
    if int(np.asarray(data["n_samples_per_family"]).item()) != int(n_samples_per_family):
        raise ValueError("Checkpoint n_samples_per_family mismatch.")
    if int(np.asarray(data["source_n_samples_per_family"]).item()) != int(source_n_samples_per_family):
        raise ValueError("Checkpoint source_n_samples_per_family mismatch.")
    if int(np.asarray(data["run_subsample_size"]).item()) != int(run_subsample_size):
        raise ValueError("Checkpoint run_subsample_size mismatch.")
    if int(np.asarray(data["run_subsample_seed"]).item()) != int(run_subsample_seed):
        raise ValueError("Checkpoint run_subsample_seed mismatch.")
    if not np.array_equal(data["input_names"].astype(str), input_names.astype(str)):
        raise ValueError("Checkpoint input_names mismatch.")
    if not np.array_equal(data["family_condition_types"].astype(str), family_condition_types.astype(str)):
        raise ValueError("Checkpoint family_condition_types mismatch.")
    if "language_codes" in data.files and not np.array_equal(
        data["language_codes"].astype(str),
        language_codes.astype(str),
    ):
        raise ValueError("Checkpoint language_codes mismatch.")
    if "language_names" in data.files and not np.array_equal(
        data["language_names"].astype(str),
        language_names.astype(str),
    ):
        raise ValueError("Checkpoint language_names mismatch.")
    if not np.array_equal(data["run_ids"].astype(np.int64), run_ids.astype(np.int64)):
        raise ValueError("Checkpoint run_ids mismatch.")
    if not np.allclose(data["context_fracs"].astype(np.float64), context_fracs.astype(np.float64)):
        raise ValueError("Checkpoint context_fracs mismatch.")
    if not np.array_equal(data["context_token_counts"].astype(np.int64), context_token_counts.astype(np.int64)):
        raise ValueError("Checkpoint context_token_counts mismatch.")
    if not np.array_equal(data["k_cutoff_by_context"].astype(np.int64), k_cutoff_by_context.astype(np.int64)):
        raise ValueError("Checkpoint k_cutoff_by_context mismatch.")

    expected_base_shape = _expected_result_shape()
    shape_checks = {
        "active_modes_layeravg_per_run": expected_base_shape,
        "fiedler_layeravg_per_run": expected_base_shape,
        "hfer_layeravg_per_run": expected_base_shape,
        "completed_mask": expected_base_shape,
    }
    for key, expected_shape in shape_checks.items():
        if data[key].shape != expected_shape:
            raise ValueError(f"Checkpoint {key} shape {data[key].shape}, expected {expected_shape}.")


def load_or_initialize_checkpoint(num_layers):
    if RESUME_FROM_CHECKPOINT and CHECKPOINT_PATH.exists():
        with np.load(CHECKPOINT_PATH, allow_pickle=False) as data:
            _validate_checkpoint_compatibility(data, num_layers=num_layers)
            arrays = {
                "active_modes_layeravg_per_run": data["active_modes_layeravg_per_run"].astype(np.float64),
                "fiedler_layeravg_per_run": data["fiedler_layeravg_per_run"].astype(np.float64),
                "hfer_layeravg_per_run": data["hfer_layeravg_per_run"].astype(np.float64),
                "completed_mask": data["completed_mask"].astype(bool),
            }
        print(
            f"Resuming checkpoint {CHECKPOINT_PATH}: "
            f"{int(arrays['completed_mask'].sum())}/{arrays['completed_mask'].size} tasks complete."
        )
        return arrays

    return _initialize_result_arrays()


# Save layer-averaged metrics and the completion mask for each language/sample/context task.
def save_checkpoint(arrays, num_layers):
    np.savez_compressed(
        CHECKPOINT_PATH,
        input_names=input_names,
        family_condition_types=family_condition_types,
        language_codes=language_codes,
        language_names=language_names,
        context_fracs=context_fracs,
        context_token_counts=context_token_counts,
        k_cutoff_by_context=k_cutoff_by_context,
        run_ids=run_ids,
        model_name=np.asarray(MODEL_NAME),
        tokenizer_name=np.asarray(tokenizer_name),
        source_sample_npz_path=np.asarray(str(SAMPLE_NPZ_PATH)),
        sample_schema_version=np.asarray(sample_schema_version),
        sample_construction_mode=np.asarray(sample_construction_mode),
        data_ordering=np.asarray("original"),
        max_context_tokens=np.asarray(max_context_tokens, dtype=np.int64),
        n_samples_per_family=np.asarray(n_samples_per_family, dtype=np.int64),
        source_n_samples_per_family=np.asarray(source_n_samples_per_family, dtype=np.int64),
        run_subsample_size=np.asarray(run_subsample_size, dtype=np.int64),
        run_subsample_seed=np.asarray(run_subsample_seed, dtype=np.int64),
        num_layers=np.asarray(num_layers, dtype=np.int64),
        active_modes_layeravg_per_run=arrays["active_modes_layeravg_per_run"],
        fiedler_layeravg_per_run=arrays["fiedler_layeravg_per_run"],
        hfer_layeravg_per_run=arrays["hfer_layeravg_per_run"],
        completed_mask=arrays["completed_mask"],
    )


Load Llama-3.2-3B with eager attention and validate the 28-layer model and signal pairing.


In [ ]:
# Eager attention returns the per-head attention matrices required for graph construction.
model_attn = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16,
    attn_implementation="eager",
)
model_attn.eval()

num_layers = int(model_attn.config.num_hidden_layers)
if num_layers != EXPECTED_NUM_LAYERS:
    raise ValueError(f"Expected model to have {EXPECTED_NUM_LAYERS} layers, got {num_layers}.")

layer_indices = np.arange(num_layers, dtype=np.int64)
layer_numbers = np.arange(1, num_layers + 1, dtype=np.int64)
if not np.array_equal(layer_numbers, np.arange(1, EXPECTED_NUM_LAYERS + 1, dtype=np.int64)):
    raise ValueError("Expected layer_numbers == 1..28.")


device = next(model_attn.parameters()).device


Sweep every language, sample, and context; average metrics over all 28 layers.


In [ ]:
arrays = load_or_initialize_checkpoint(num_layers=num_layers)
completed_since_checkpoint = 0

# One task is one language, sample, and content-token context.
tasks = [
    (input_axis_idx, str(data_name), run_axis_idx, int(rep_id), context_idx)
    for input_axis_idx, data_name in enumerate(input_names)
    for run_axis_idx, rep_id in enumerate(run_ids)
    for context_idx in range(len(context_token_counts))
]

checkpoint_every = max(1, int(CHECKPOINT_EVERY_N_TASKS))
initial_completed_total = int(arrays["completed_mask"].sum())
next_checkpoint_completed = (
    (initial_completed_total // checkpoint_every) + 1
) * checkpoint_every

for input_axis_idx, data_name, run_axis_idx, rep_id, context_idx in tqdm(
    tasks,
    desc="MultiUN language all metrics layer-averaged vs context",
    miniters=checkpoint_every,
):
    if arrays["completed_mask"][input_axis_idx, run_axis_idx, context_idx]:
        continue

    sample_ids = token_ids[input_axis_idx, run_axis_idx, :]
    expected_seq_len = int(context_token_counts[context_idx])
    expected_k_cutoff = int(k_cutoff_by_context[context_idx])

    built = build_model_inputs_from_saved_token_ids(
        sample_ids,
        expected_seq_len,
        tokenizer=tokenizer,
        device=device,
        prepend_bos=PREPEND_BOS,
    )
    if int(built["seq_len"]) != expected_seq_len:
        raise ValueError("Built model input length must equal context_token_counts.")
    if int(built["token_start"]) != 1 and PREPEND_BOS and DROP_BOS:
        raise ValueError("Expected the prepended BOS token to be dropped before metric computation.")

    model_inputs = built["model_inputs"]
    input_ids_for_metrics = built["input_ids_for_metrics"]
    special_tokens_mask = built["special_tokens_mask"]

    # Obtain all attention and hidden-state tensors in one forward pass for this context.
    with torch.no_grad():
        outputs = model_attn(
            **model_inputs,
            output_attentions=True,
            output_hidden_states=True,
            return_dict=True,
        )

    if outputs.attentions is None or outputs.hidden_states is None:
        raise ValueError("Model outputs must include attentions and hidden_states.")
    if len(outputs.attentions) != num_layers:
        raise ValueError(f"Expected {num_layers} attention tensors, got {len(outputs.attentions)}.")
    if len(outputs.hidden_states) != num_layers + 1:
        raise ValueError(f"Expected {num_layers + 1} hidden-state tensors, got {len(outputs.hidden_states)}.")

    active_modes_sum = 0.0
    fiedler_sum = 0.0
    hfer_sum = 0.0

    # hidden_states[layer_idx] is the input to the corresponding attention layer.
    # This includes the embedding signal for layer 0 and excludes the final output state.
    for layer_idx in range(num_layers):
        bundle = compute_all_layer_metrics_from_attention_and_hidden(
            outputs.attentions[layer_idx],
            outputs.hidden_states[layer_idx],
            input_ids=input_ids_for_metrics,
            layer_idx=layer_idx,
            expected_seq_len=expected_seq_len,
            k_cutoff=expected_k_cutoff,
            bos_token_id=tokenizer.bos_token_id,
            special_tokens_mask=special_tokens_mask,
            drop_bos=DROP_BOS,
            renormalize_after_drop=RENORMALIZE_AFTER_DROP,
            head_weighting=HEAD_WEIGHTING,
        )
        if int(bundle["attention_layer_index"]) != layer_idx or int(bundle["signal_layer_index"]) != layer_idx:
            raise ValueError("Layer pairing failed: attention and signal layer indices must equal layer_idx.")
        if int(bundle["seq_len"]) != expected_seq_len:
            raise ValueError("Graph seq_len must equal the selected context_token_counts value.")
        if int(bundle["k_cutoff"]) != expected_k_cutoff:
            raise ValueError("Layer k_cutoff must equal context_token_counts // 2.")


        active_modes_sum += float(bundle["effective_active_modes"])
        fiedler_sum += float(bundle["normalized_fiedler_value"])
        hfer_sum += float(bundle["hfer"])
        del bundle

    # Average metrics over layers within this sample/context.
    active_modes_layeravg = active_modes_sum / num_layers
    fiedler_layeravg = fiedler_sum / num_layers
    hfer_layeravg = hfer_sum / num_layers


    arrays["active_modes_layeravg_per_run"][input_axis_idx, run_axis_idx, context_idx] = active_modes_layeravg
    arrays["fiedler_layeravg_per_run"][input_axis_idx, run_axis_idx, context_idx] = fiedler_layeravg
    arrays["hfer_layeravg_per_run"][input_axis_idx, run_axis_idx, context_idx] = hfer_layeravg
    arrays["completed_mask"][input_axis_idx, run_axis_idx, context_idx] = True
    completed_since_checkpoint += 1

    # Release model outputs before moving to the next context.
    del outputs, model_inputs, built, input_ids_for_metrics, special_tokens_mask
    del active_modes_sum, fiedler_sum, hfer_sum
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    completed_total = int(arrays["completed_mask"].sum())
    if completed_total >= next_checkpoint_completed:
        save_checkpoint(arrays, num_layers=num_layers)
        completed_since_checkpoint = 0
        next_checkpoint_completed = (
            (completed_total // checkpoint_every) + 1
        ) * checkpoint_every

if completed_since_checkpoint > 0 or not CHECKPOINT_PATH.exists():
    save_checkpoint(arrays, num_layers=num_layers)

print(f"Sweep complete: {int(arrays['completed_mask'].sum())}/{arrays['completed_mask'].size} tasks.")


Define sample-summary statistics and validation checks for the completed metric arrays.


In [ ]:
def _summarize_over_runs(per_run):
    # Summarize the sample axis; use sample standard deviation (ddof=1) and SEM.
    mean = np.mean(per_run, axis=1)
    if per_run.shape[1] > 1:
        std = np.std(per_run, axis=1, ddof=1)
    else:
        std = np.zeros_like(mean)
    sem = std / np.sqrt(per_run.shape[1])
    return mean, std, sem


# Check finiteness and metric ranges.
def _validate_metric_block(prefix, active, fiedler, hfer):
    base_shape = _expected_result_shape()
    expected_shapes = {
        f"active_modes_{prefix}_per_run": (active, base_shape),
        f"fiedler_{prefix}_per_run": (fiedler, base_shape),
        f"hfer_{prefix}_per_run": (hfer, base_shape),
    }
    for key, (value, expected_shape) in expected_shapes.items():
        if value.shape != expected_shape:
            raise ValueError(f"{key} shape {value.shape}, expected {expected_shape}.")
        if not np.isfinite(value).all():
            bad = np.argwhere(~np.isfinite(value))[:10]
            raise ValueError(f"{key} contains nonfinite values; first bad indices: {bad}.")

    if not np.all(active > 0):
        raise ValueError(f"{prefix} active modes must be positive.")
    if not np.all((hfer >= -1e-6) & (hfer <= 1.0 + 1e-6)):
        raise ValueError(f"{prefix} HFER values must be in [0, 1] up to tiny numerical tolerance.")
    if not np.all((fiedler >= -1e-4) & (fiedler <= 2.0 + 1e-4)):
        raise ValueError(f"{prefix} normalized Fiedler values must be in [0, 2] up to numerical tolerance.")


def _validate_final_arrays():
    if not np.all(arrays["completed_mask"]):
        bad = np.argwhere(~arrays["completed_mask"])[:10]
        raise ValueError(f"Not all tasks are complete; first incomplete indices: {bad}.")

    _validate_metric_block(
        "layeravg",
        arrays["active_modes_layeravg_per_run"],
        arrays["fiedler_layeravg_per_run"],
        arrays["hfer_layeravg_per_run"],
    )

    if not np.array_equal(k_cutoff_by_context, context_token_counts // 2):
        raise ValueError("k_cutoff_by_context must equal context_token_counts // 2.")


    if not np.all(np.diff(context_token_counts) > 0):
        raise ValueError("context_token_counts must be strictly increasing.")
    if int(context_token_counts[-1]) != int(max_context_tokens):
        raise ValueError("Final context length must equal max_context_tokens post-BOS content tokens.")


Validate all results and calculate means, sample standard deviations, and standard errors across samples.


In [ ]:
# Require every task to be complete before creating the final summaries.
_validate_final_arrays()

# Summarize layer averages over the 50 samples in each language.
active_modes_mean, active_modes_std, active_modes_sem = _summarize_over_runs(
    arrays["active_modes_layeravg_per_run"]
)
fiedler_mean, fiedler_std, fiedler_sem = _summarize_over_runs(
    arrays["fiedler_layeravg_per_run"]
)
hfer_mean, hfer_std, hfer_sem = _summarize_over_runs(
    arrays["hfer_layeravg_per_run"]
)


expected_summary_shape = (len(input_names), len(context_token_counts))
for key, value, expected_shape in [
    ("active_modes_mean", active_modes_mean, expected_summary_shape),
    ("active_modes_std", active_modes_std, expected_summary_shape),
    ("active_modes_sem", active_modes_sem, expected_summary_shape),
    ("fiedler_mean", fiedler_mean, expected_summary_shape),
    ("fiedler_std", fiedler_std, expected_summary_shape),
    ("fiedler_sem", fiedler_sem, expected_summary_shape),
    ("hfer_mean", hfer_mean, expected_summary_shape),
    ("hfer_std", hfer_std, expected_summary_shape),
    ("hfer_sem", hfer_sem, expected_summary_shape),
]:
    if value.shape != expected_shape:
        raise ValueError(f"{key} shape {value.shape}, expected {expected_shape}.")
    if not np.isfinite(value).all():
        raise ValueError(f"{key} contains nonfinite values.")


Save the layer-averaged diagnostic archive with its relative source path.


In [ ]:
np.savez_compressed(
    OUTPUT_PATH,
    input_names=input_names,
    context_token_counts=context_token_counts,
    run_ids=run_ids,
    num_layers=np.asarray(num_layers, dtype=np.int64),
    num_ensemble_runs=np.asarray(len(run_ids), dtype=np.int64),
    model_name=np.asarray(MODEL_NAME),
    metric_layer_mode=np.asarray("layer_averaged"),
    active_modes_mean=active_modes_mean,
    active_modes_std=active_modes_std,
    active_modes_sem=active_modes_sem,
    fiedler_mean=fiedler_mean,
    fiedler_std=fiedler_std,
    fiedler_sem=fiedler_sem,
    hfer_mean=hfer_mean,
    hfer_std=hfer_std,
    hfer_sem=hfer_sem,
    active_modes_layeravg_per_run=arrays["active_modes_layeravg_per_run"],
    fiedler_layeravg_per_run=arrays["fiedler_layeravg_per_run"],
    hfer_layeravg_per_run=arrays["hfer_layeravg_per_run"],
    # Record the source path relative to the appendix root in the final archive.
    source_sample_npz_path=np.asarray(os.path.relpath(SAMPLE_NPZ_PATH, start=APPENDIX_DIR)),
    tokenizer_name=np.asarray(tokenizer_name),
    sample_schema_version=np.asarray(sample_schema_version),
    sample_construction_mode=np.asarray(sample_construction_mode),
    family_condition_types=family_condition_types,
    language_codes=language_codes,
    language_names=language_names,
    max_context_tokens=np.asarray(max_context_tokens, dtype=np.int64),
    n_samples_per_family=np.asarray(n_samples_per_family, dtype=np.int64),
    source_n_samples_per_family=np.asarray(source_n_samples_per_family, dtype=np.int64),
    run_subsample_size=np.asarray(run_subsample_size, dtype=np.int64),
    run_subsample_seed=np.asarray(run_subsample_seed, dtype=np.int64),
    data_ordering=np.asarray("original"),
    context_fracs=context_fracs,
    k_cutoff_by_context=k_cutoff_by_context,
)